# 03 Matching Comparison — cost-sensitive SKU matching benchmark

Эта тетрадка начинается после ручной разметки. На входе у нас есть CSV с парами товаров и твоими метками: тот же это базовый товар или другой товар.

Главный production-вопрос теперь не `какой macro-F1 выше`, а `какую долю пар можно автоматически решить без опасных false merge`.

Для этого тетрадка:

- считает `score` для rule-based, bi-encoder, cross-encoder и выбранных reranker-моделей;
- строит binary target `same_base_product` (`exact_duplicate` и legacy `same_product_different_pack` = 1, `different_product` = 0);
- на `dev` отдельно калибрует `threshold_auto_same` и `threshold_auto_diff`;
- применяет эти пороги к `test` без подбора на test;
- сохраняет calibration/evaluation/error reports в `artifacts/reports/`.

Фасовка и multipack не являются отдельным ML-классом: они разбираются после модели deterministic правилами.


## Мини-словарь перед запуском

`same_base_product` — бинарный таргет для модели: `1`, если это тот же базовый товар, и `0`, если это другой товар.

`threshold_auto_same` — высокий порог для безопасного auto-merge: `score >= threshold_auto_same`.

`threshold_auto_diff` — низкий порог для безопасного auto-reject: `score <= threshold_auto_diff`.

`manual_review` — промежуточная зона между двумя порогами. Эти пары не склеиваем и не отклоняем автоматически.

`false merge` — самая дорогая ошибка: настоящий `different_product` попал в `auto_same`.

`false reject` — менее дорогая ошибка: настоящий same-base товар попал в `auto_different`.

`macro-F1` можно смотреть как общий sanity-check, но он больше не является критерием выбора auto-merge порога или модели.


## Как устроена проверка

Мы делим размеченные пары на две части:

- `dev` — часть для подбора обоих порогов. Только здесь выбираются thresholds.
- `test` — отложенная часть для честной проверки. На ней запрещено выбирать threshold или модель.

Auto-merge калибруется консервативно: максимум recall среди threshold-ов, которые проходят precision constraint и лимит false merges на `dev`.


## Блок кода 1. Подготовка окружения

Эта ячейка подключает библиотеки, находит корень проекта и импортирует нужные функции из `research/dedup`.

Если здесь ошибка, чаще всего причина простая: тетрадка запущена не из папки проекта или не установлен пакет для ноутбуков.


In [ ]:
from __future__ import annotations

from pathlib import Path
import os
import sys
import time
from typing import Any

from IPython.display import display
import pandas as pd

try:
    import matplotlib.pyplot as plt
except ImportError:  # pragma: no cover - notebook environment dependent
    plt = None

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from research.dedup import (
    CROSS_ENCODER_BACKEND,
    SENTENCE_TRANSFORMER_BACKEND,
    TRANSFORMERS_AUTO_MODEL_BACKEND,
    BiEncoderMatcher,
    FusionConfig,
    ModelManager,
    POLZA_EMBEDDING_BACKEND,
    RuleBasedMatcher,
    ThresholdCalibrationConfig,
    calibrate_and_evaluate_methods,
    forced_confusion_matrix,
    prepare_calibration_frame,
    same_base_product_target,
    triage_confusion_matrix,
    write_threshold_reports,
)
from research.dedup.matchers.bi_encoder import BiEncoderConfig

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 160)


## Блок кода 2. Настройки тетрадки

Здесь задаются пути к файлам и основные параметры проверки.

Самое важное:

- `LABELING_PATH` — файл с ручной разметкой.
- `RUN_BI_ENCODER` — запускать ли модель для сравнения текстов.
- `BI_ENCODER_MODEL` — alias из `research.dedup.model_registry` или прямой model id, например `openai/text-embedding-3-small`.
- `DEDUP_BI_ENCODER_BACKEND=polza_embedding` — только для нового прямого Polza model id, которого ещё нет в registry; известные Polza ids распознаются автоматически.
- `POLZA_API_KEY` или `POLZA_AI_API_KEY` — ключ для online-моделей Polza.ai.
- `POLZA_BASE_URL` — override base URL, по умолчанию `https://polza.ai/api/v1`.
- `DEDUP_MODEL_CACHE_DIR` — куда локально складываются скачанные local-модели; по умолчанию `research/dedup/models/`.
- `DEDUP_MODEL_LOCAL_ONLY=1` — offline-режим: не скачивать модель, а брать только уже лежащую в кэше.
- `DEV_FRACTION` — какая часть размеченных пар идёт на подбор порогов.
- `TARGET_AUTO_SAME_PRECISION` — минимальная precision для автоматической склейки.
- `MAX_FALSE_MERGES_ON_DEV` — максимально допустимое число false merges на dev.
- `TARGET_AUTO_DIFF_PRECISION` — минимальная precision для автоматического отклонения.

Если у тебя не установлен `sentence-transformers`, метод `bi_encoder_zero_shot` не сможет нормально отработать.


In [ ]:
DATA_DIR = PROJECT_ROOT / "research" / "dedup" / "data"
REPORTS_DIR = PROJECT_ROOT / "artifacts" / "reports"
LABELING_PATH = DATA_DIR / "labeling_sauces.csv"
PREDICTIONS_PATH = DATA_DIR / "matching_predictions_sauces.csv"
SUMMARY_PATH = DATA_DIR / "matching_summary_sauces.csv"
FALSE_MERGES_PATH = DATA_DIR / "matching_false_merges_sauces.csv"

SOURCE_LABELS = ["exact_duplicate", "same_product_different_pack", "different_product"]

MODEL_MANAGER = ModelManager()
RUN_BI_ENCODER = os.environ.get("DEDUP_RUN_BI_ENCODER", "1") == "1"
BI_ENCODER_MODEL = os.environ.get("DEDUP_BI_ENCODER_MODEL", "bi_encoder_e5_small")
BI_ENCODER_MODEL_BACKEND = os.environ.get("DEDUP_BI_ENCODER_BACKEND", "").strip() or None
BI_ENCODER_MODEL_SPEC = MODEL_MANAGER.resolve_embedding_model(BI_ENCODER_MODEL, backend=BI_ENCODER_MODEL_BACKEND)
RANDOM_STATE = int(os.environ.get("DEDUP_EVAL_RANDOM_STATE", "42"))
DEV_FRACTION = float(os.environ.get("DEDUP_EVAL_DEV_FRACTION", "0.60"))
TARGET_AUTO_SAME_PRECISION = float(os.environ.get("DEDUP_TARGET_AUTO_SAME_PRECISION", "0.97"))
MAX_FALSE_MERGES_ON_DEV = int(os.environ.get("DEDUP_MAX_FALSE_MERGES_ON_DEV", "0"))
TARGET_AUTO_DIFF_PRECISION = float(os.environ.get("DEDUP_TARGET_AUTO_DIFF_PRECISION", "0.95"))
THRESHOLD_CONFIG = ThresholdCalibrationConfig(
    target_auto_same_precision=TARGET_AUTO_SAME_PRECISION,
    max_false_merges_on_dev=MAX_FALSE_MERGES_ON_DEV,
    target_auto_diff_precision=TARGET_AUTO_DIFF_PRECISION,
)

print(f"Labeling path: {LABELING_PATH}")
print(f"Reports dir: {REPORTS_DIR}")
print(f"Run bi-encoder: {RUN_BI_ENCODER}")
print(f"Bi-encoder model alias/input: {BI_ENCODER_MODEL}")
print(f"Bi-encoder model id: {BI_ENCODER_MODEL_SPEC.model_name}")
print(f"Bi-encoder backend: {BI_ENCODER_MODEL_SPEC.backend}")
if BI_ENCODER_MODEL_SPEC.backend == POLZA_EMBEDDING_BACKEND:
    print(f"Polza base URL: {MODEL_MANAGER.polza_base_url}")
print(f"Model cache dir: {MODEL_MANAGER.cache_dir}")
print(f"Local-only model loading: {MODEL_MANAGER.local_files_only}")
print(f"Dev fraction: {DEV_FRACTION:.0%}; random_state={RANDOM_STATE}")
print(f"Target auto-same precision: {TARGET_AUTO_SAME_PRECISION:.0%}; max false merges on dev: {MAX_FALSE_MERGES_ON_DEV}")
print(f"Target auto-diff precision: {TARGET_AUTO_DIFF_PRECISION:.0%}")


## Блок кода 3. Загрузка и первичная проверка разметки

Эта ячейка читает `labeling_sauces.csv`, проверяет колонку `label`, убирает `uncertain` из расчёта метрик и делит строки на `dev` и `test`.

В выводе нужно смотреть:

- сколько всего строк в разметке;
- сколько строк реально попало в метрики;
- сколько строк отброшено как `uncertain`;
- как классы распределились между `dev` и `test`.

Если классов в `test` очень мало, итоговые цифры будут шумными.


In [ ]:
def load_labeled_pairs(path: Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    if not path.exists():
        status = pd.DataFrame([
            {
                "status": "missing_labeling_file",
                "message": f"Файл {path} пока не найден. Выполните 02_labeling_dataset.ipynb и заполните label.",
                "rows_total": 0,
                "rows_usable_for_metrics": 0,
                "rows_ignored_without_binary_target": 0,
            }
        ])
        return pd.DataFrame(), status

    frame = pd.read_csv(path)
    if "label" not in frame.columns and "same_base_product" not in frame.columns:
        status = pd.DataFrame([
            {
                "status": "missing_target_columns",
                "message": "В файле нет ни label, ни same_base_product. Перегенерируйте labeling dataset из notebook-2.",
                "rows_total": len(frame),
                "rows_usable_for_metrics": 0,
                "rows_ignored_without_binary_target": len(frame),
            }
        ])
        return pd.DataFrame(), status

    target = same_base_product_target(frame)
    labeled = frame[target.notna()].copy()
    labeled["same_base_product"] = target[target.notna()].astype(int).to_numpy()
    if "label" in labeled.columns:
        labeled["label"] = labeled["label"].fillna("").astype(str).str.strip()
    ignored_count = int(len(frame) - len(labeled))
    status_name = "ready" if not labeled.empty else "empty_or_not_reviewed_yet"
    message = (
        "Gold-set готов для cost-sensitive calibration."
        if not labeled.empty
        else "Binary target пока не заполнен: метрики ниже будут заглушками, notebook не падает."
    )
    status = pd.DataFrame([
        {
            "status": status_name,
            "message": message,
            "rows_total": len(frame),
            "rows_usable_for_metrics": len(labeled),
            "rows_ignored_without_binary_target": ignored_count,
        }
    ])
    return labeled.reset_index(drop=True), status


def add_stratified_eval_split(frame: pd.DataFrame) -> pd.DataFrame:
    if frame.empty:
        return frame.assign(eval_split=pd.Series(dtype="string"))
    parts: list[pd.DataFrame] = []
    for _, group in frame.groupby("same_base_product", sort=False):
        shuffled = group.sample(frac=1.0, random_state=RANDOM_STATE)
        if len(shuffled) == 1:
            dev = shuffled.copy()
            dev["eval_split"] = "dev"
            parts.append(dev)
            continue
        dev_count = int(round(len(shuffled) * DEV_FRACTION))
        dev_count = min(max(1, dev_count), len(shuffled) - 1)
        dev = shuffled.iloc[:dev_count].copy()
        test = shuffled.iloc[dev_count:].copy()
        dev["eval_split"] = "dev"
        test["eval_split"] = "test"
        parts.extend([dev, test])
    return pd.concat(parts).sort_index().reset_index(drop=True)


labeled_pairs, labeling_status = load_labeled_pairs(LABELING_PATH)
labeled_pairs = add_stratified_eval_split(labeled_pairs)

display(labeling_status)
if labeled_pairs.empty:
    display(pd.DataFrame(columns=["same_base_product", "pairs"]))
else:
    display(labeled_pairs["same_base_product"].map({1: "same_base_product", 0: "different_product"}).value_counts().rename_axis("target").reset_index(name="pairs"))
    display(pd.crosstab(labeled_pairs["eval_split"], labeled_pairs["same_base_product"].map({1: "same_base_product", 0: "different_product"})))
    if "label" in labeled_pairs.columns:
        display(labeled_pairs["label"].value_counts().rename_axis("source_label").reset_index(name="pairs"))


## Блок кода 4. Список методов, которые будем сравнивать

Эта ячейка создаёт методы сравнения пар и показывает, доступны ли они в текущем окружении.

Важно смотреть на строку `bi_encoder_zero_shot`:

- `available=True` означает, что пакет найден;
- `available=False` означает, что модельный способ будет пропущен.

Если метод пропущен, это не портит rule-based проверку, но сравнение с моделью будет неполным.


In [ ]:
matchers = [RuleBasedMatcher()]
if RUN_BI_ENCODER:
    matchers.append(BiEncoderMatcher(BiEncoderConfig(model_name=BI_ENCODER_MODEL, model_backend=BI_ENCODER_MODEL_BACKEND)))

method_status = []
for matcher in matchers:
    status = matcher.status()
    method_status.append({"method": matcher.name, "available": status.available, "status": status.message})

method_status_df = pd.DataFrame(method_status)
display(method_status_df)


## Блок кода 5. Вспомогательные функции для scoring

Эта ячейка не выбирает пороги. Она только задаёт общий способ посчитать `score` для пары и собрать score-таблицу.

Пороговая логика живёт ниже в cost-sensitive calibration block.


In [ ]:
def _score_matcher(matcher: Any, pairs: pd.DataFrame) -> tuple[list[float], str]:
    if pairs.empty:
        return [], "skipped_empty_gold_set"
    row_objects = [row for _, row in pairs.iterrows()]
    score_batch = getattr(matcher, "score_batch", None)
    if callable(score_batch):
        scores = score_batch(row_objects)
    else:
        scores = [matcher.score(row) for row in row_objects]
    if scores and all(pd.isna(score) for score in scores):
        return scores, matcher.status().message
    return scores, "ready"


def _with_benchmark_pair_key(frame: pd.DataFrame) -> pd.DataFrame:
    output = frame.copy()
    if {"raw_record_id_a", "raw_record_id_b"}.issubset(output.columns):
        left_values = output["raw_record_id_a"].astype(str)
        right_values = output["raw_record_id_b"].astype(str)
    else:
        left_values = output.get("title_a", pd.Series([""] * len(output))).astype(str)
        right_values = output.get("title_b", pd.Series([""] * len(output))).astype(str)
    output["benchmark_pair_key"] = [
        " || ".join(sorted([left, right]))
        for left, right in zip(left_values, right_values, strict=False)
    ]
    return output


def _score_frame(method: str, frame: pd.DataFrame, scores: list[float], *, benchmark_source: str) -> pd.DataFrame:
    output = _with_benchmark_pair_key(frame)
    output["method"] = method
    output["score"] = scores
    output["benchmark_source"] = benchmark_source
    return output


## Блок кода 6. Подсчёт сходства для всех пар

Эта ячейка прогоняет каждый метод по размеченным парам и сохраняет численные оценки `score`.

Что смотреть в выводе:

- `status=ready` — метод отработал;
- `seconds` — сколько времени занял расчёт.

Для rule-based обычно всё быстро. Для `bi_encoder_zero_shot` может быть дольше, потому что загружается модель и считаются векторы текстов.


In [6]:
scored_methods: dict[str, dict[str, object]] = {}
skipped_methods: list[dict[str, str]] = []

if labeled_pairs.empty:
    display(pd.DataFrame([{"method": "not_available_yet", "status": labeling_status.loc[0, "status"], "seconds": 0.0}]))
else:
    scoring_rows = []
    for matcher in matchers:
        started = time.perf_counter()
        scores, status = _score_matcher(matcher, labeled_pairs)
        elapsed = time.perf_counter() - started
        scoring_rows.append({"method": matcher.name, "status": status, "seconds": round(elapsed, 3)})
        if status != "ready":
            skipped_methods.append({"method": matcher.name, "status": status})
            continue
        scored_methods[matcher.name] = {"matcher": matcher, "scores": scores, "seconds": elapsed}
    display(pd.DataFrame(scoring_rows))
    if skipped_methods:
        display(pd.DataFrame(skipped_methods))


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5282.57it/s]

,method,status,seconds
0,rule_based_fuzzy,ready,0.039
1,bi_encoder_zero_shot,ready,21.303


## Блок кода 7. Быстрый score sanity-check

Эта ячейка не выбирает threshold. Она только показывает диапазон score по методам, чтобы сразу увидеть пустые/битые прогоны.


In [ ]:
score_overview_rows: list[dict[str, object]] = []

for method, payload in scored_methods.items():
    score_series = pd.to_numeric(pd.Series(payload["scores"]), errors="coerce").dropna()
    score_overview_rows.append(
        {
            "method": method,
            "pairs": len(payload["scores"]),
            "score_min": float(score_series.min()) if not score_series.empty else None,
            "score_median": float(score_series.median()) if not score_series.empty else None,
            "score_max": float(score_series.max()) if not score_series.empty else None,
            "seconds": round(float(payload.get("seconds", 0.0)), 3),
        }
    )

if score_overview_rows:
    display(pd.DataFrame(score_overview_rows))
else:
    display(pd.DataFrame([{"method": "not_available_yet", "pairs": 0}]))


## Блок кода 8. Cost-sensitive calibration выполняется после scoring всех моделей

Раньше здесь подбирался один threshold по `macro_f1`. Этот блок намеренно убран из основной логики: сначала ниже считаются cross-encoder/reranker scores, затем единый финальный block калибрует все methods одинаково.


In [ ]:
calibration_df = pd.DataFrame()
calibrated_summary_df = pd.DataFrame()
all_calibrated_predictions = pd.DataFrame()
summary_export = pd.DataFrame()

print("Threshold selection moved to the final cost-sensitive benchmark block.")


## Блок кода 9. Матрицы ошибок теперь строятся для forced и triage predictions

Confusion matrices выводятся в финальном cost-sensitive block после выбора `threshold_auto_same` и `threshold_auto_diff` на dev.


In [ ]:
print("Confusion matrices will be displayed after cost-sensitive threshold calibration.")


## Блок кода 10. Сохранение результатов перенесено в финальный benchmark block

Финальный block сохраняет:

- `artifacts/reports/threshold_calibration_dev.csv`;
- `artifacts/reports/threshold_evaluation_test.csv`;
- error reports для false merges, false rejects и manual review;
- совместимые `research/dedup/data/matching_*` CSV для следующих тетрадок.


In [ ]:
print("Exports will be written after all selected methods are scored.")


## Что делать после этой тетрадки

Если результат `rule_based_fuzzy` и `bi_encoder_zero_shot` слабый, это нормально для первого прогона. Эта тетрадка нужна не для финальной победы, а чтобы увидеть нижнюю планку и понять, где методы ошибаются.

Главный вывод сейчас: похожесть текста сама по себе часто путает разные вкусы и типы соусов. Следующий разумный шаг — добавить более сильную проверку пары, например cross-encoder или отдельную проверку спорных случаев.


## Новая итерация: добавляем готовый cross-encoder

Предыдущие блоки показали важную проблему: простая похожесть названий и обычные векторы часто путают похожие, но разные товары.

Теперь добавляем следующий метод из архитектуры: cross-encoder. Он читает пару товаров вместе: товар A и товар B одновременно. Поэтому он теоретически должен лучше замечать различия вроде `сырный` против `барбекю` или `сальса` против `сладкий чили`.

Пока это не обученная на наших данных модель, а готовая модель из `sentence-transformers`. Поэтому это промежуточный опыт: проверяем, помогает ли более внимательное сравнение пары даже без дообучения.


## Блок кода 11. Настройки cross-encoder

Эта ячейка добавляет новый метод, но не трогает старые результаты выше.

Что важно:

- `DEDUP_RUN_CROSS_ENCODER=0` можно поставить, если нужно временно пропустить этот блок.
- `DEDUP_CROSS_ENCODER_MODEL` теперь принимает alias из model registry или прямой Hugging Face model id.
- По умолчанию используется alias `cross_encoder_mmarco`, который указывает на `cross-encoder/mmarco-mMiniLMv2-L12-H384-v1`.
- Скачивание и повторное использование модели управляется через `ModelManager`: кэш `DEDUP_MODEL_CACHE_DIR`, offline-флаг `DEDUP_MODEL_LOCAL_ONLY=1`.

Если модель ещё не скачана, первый запуск может занять время. Если интернет недоступен, включи offline-режим только после предварительного прогрева кэша.


In [ ]:
from research.dedup import CrossEncoderMatcher
from research.dedup.matchers.cross_encoder import CrossEncoderConfig

RUN_CROSS_ENCODER = os.environ.get("DEDUP_RUN_CROSS_ENCODER", "1") == "1"
CROSS_ENCODER_MODEL = os.environ.get("DEDUP_CROSS_ENCODER_MODEL", "cross_encoder_mmarco")
CROSS_ENCODER_MODEL_SPEC = MODEL_MANAGER.resolve(CROSS_ENCODER_MODEL, backend=CROSS_ENCODER_BACKEND)
CROSS_ENCODER_BATCH_SIZE = int(
    os.environ.get("DEDUP_CROSS_ENCODER_BATCH_SIZE", str(CROSS_ENCODER_MODEL_SPEC.batch_size or 16))
)

print(f"Run cross-encoder: {RUN_CROSS_ENCODER}")
print(f"Cross-encoder model alias/input: {CROSS_ENCODER_MODEL}")
print(f"Cross-encoder model id: {CROSS_ENCODER_MODEL_SPEC.model_name}")
print(f"Cross-encoder batch size: {CROSS_ENCODER_BATCH_SIZE}")
print(f"Model cache dir: {MODEL_MANAGER.cache_dir}")
print(f"Local-only model loading: {MODEL_MANAGER.local_files_only}")


## Блок кода 12. Запуск cross-encoder на тех же парах

Эта ячейка считает `score` для каждой размеченной пары.

`score` здесь означает: насколько готовая модель считает пару подходящей для связи. У этой модели score не обязан быть от 0 до 1: важен не абсолютный смысл числа, а то, как оно разделяет хорошие и плохие пары. Потом этот score проходит через ту же простую логику с весом, фасовкой и брендом, чтобы получить один из трёх классов.

Мы используем те же `dev` и `test`, что выше. Это важно: новый метод сравнивается на той же отложенной части, а не на новых случайных строках.


In [12]:
cross_encoder_payload: dict[str, object] | None = None
cross_encoder_status_rows: list[dict[str, object]] = []

if not RUN_CROSS_ENCODER:
    cross_encoder_status_rows.append({"method": "cross_encoder_zero_shot", "status": "skipped_by_env", "seconds": 0.0})
elif labeled_pairs.empty:
    cross_encoder_status_rows.append({"method": "cross_encoder_zero_shot", "status": "skipped_empty_gold_set", "seconds": 0.0})
else:
    cross_encoder = CrossEncoderMatcher(
        CrossEncoderConfig(
            model_name=CROSS_ENCODER_MODEL,
            batch_size=CROSS_ENCODER_BATCH_SIZE,
        )
    )
    started = time.perf_counter()
    cross_scores, cross_status = _score_matcher(cross_encoder, labeled_pairs)
    elapsed = time.perf_counter() - started
    cross_encoder_status_rows.append(
        {"method": cross_encoder.name, "status": cross_status, "seconds": round(elapsed, 3)}
    )
    if cross_status == "ready":
        cross_encoder_payload = {"matcher": cross_encoder, "scores": cross_scores, "seconds": elapsed}

cross_encoder_status_df = pd.DataFrame(cross_encoder_status_rows)
display(cross_encoder_status_df)


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 6401.09it/s]

,method,status,seconds
0,cross_encoder_zero_shot,ready,27.663


## Блок кода 13. Cross-encoder score готовится для общей калибровки

Эта ячейка больше не подбирает отдельный macro-F1 threshold. Если cross-encoder успешно посчитан, он попадёт в финальный cost-sensitive benchmark вместе с остальными methods.


In [ ]:
if cross_encoder_payload is None:
    display(pd.DataFrame([{"method": "cross_encoder_zero_shot", "status": "not_available"}]))
else:
    score_series = pd.to_numeric(pd.Series(cross_encoder_payload["scores"]), errors="coerce").dropna()
    display(pd.DataFrame([
        {
            "method": cross_encoder_payload["matcher"].name,
            "status": "ready",
            "pairs": len(cross_encoder_payload["scores"]),
            "score_min": float(score_series.min()) if not score_series.empty else None,
            "score_median": float(score_series.median()) if not score_series.empty else None,
            "score_max": float(score_series.max()) if not score_series.empty else None,
            "seconds": round(float(cross_encoder_payload.get("seconds", 0.0)), 3),
        }
    ]))


Cost-sensitive evaluation ниже покажет, уменьшает ли cross-encoder false merges при строгом auto-merge пороге.


In [ ]:
print("Cross-encoder confusion matrices will be displayed in the final cost-sensitive benchmark block.")


## Блок кода 15. Обновление CSV выполняется финальным cost-sensitive block

Cross-encoder не пишет отдельные calibrated CSV: все methods сохраняются единым набором артефактов после общей калибровки.


In [ ]:
print("CSV export is handled by the final cost-sensitive benchmark block.")


## Что мы проверяем этой новой итерацией

Эта секция отвечает на конкретный вопрос: помогает ли готовый cross-encoder как более внимательная проверка пары.

Если он заметно снижает false merges на `test`, это сильный аргумент двигаться в сторону cross-encoder rerank.

Если он не помогает достаточно, это тоже нормальный результат: тогда показываем жюри, что готовой модели мало, и нужен следующий шаг — дообучение на наших парах или отдельный LLM-judge для спорных случаев.


## Общий бенчмарк всех matching-моделей

Эта секция сравнивает текущие baseline-методы и выбранные reranker-модели на одном и том же наборе размеченных пар:

- `rule_based_fuzzy`;
- `bi_encoder_zero_shot`;
- `cross_encoder_zero_shot`;
- `BAAI/bge-reranker-v2-m3` по умолчанию;
- опционально `Qwen/Qwen3-Reranker-4B`, `Qwen/Qwen3-Reranker-0.6B` и `jinaai/jina-reranker-v3`.

Запуск обычный: меняете настройки в следующей ячейке и жмёте Run. Никакие переменные окружения для включения блока не нужны.

Первый запуск может быть долгим именно на скачивании модели из Hugging Face. `Qwen/Qwen3-Reranker-4B` тяжёлый и на Mac может не помещаться в MPS, поэтому registry запускает его на CPU. Если нужен быстрый Qwen-smoke, попробуйте alias `qwen3_0_6b`.

## Блок кода 16. Настройки общего бенчмарка

Главное место для настройки моделей — верх следующей code-ячейки, блок `НАСТРОЙКИ ПОЛЬЗОВАТЕЛЯ`.

Туда вписываются новые reranker-модели, которые добавляются к уже посчитанным выше baseline-методам. Старые методы (`rule_based_fuzzy`, `bi_encoder_zero_shot`, `cross_encoder_zero_shot`) перечислять не нужно: они подтягиваются из предыдущих блоков этой же тетрадки автоматически.

Что менять чаще всего:

- `MY_RERANKER_MODELS` — список моделей для запуска. Можно писать короткие alias-ы (`"bge_m3"`, `"qwen3_4b"`, `"jina_v3"`) или полные Hugging Face model ids (`"BAAI/bge-reranker-v2-m3"`).
- `MY_RERANKER_MAX_PAIRS` — размер быстрого среза. `120` удобно для первого прогона, `0` означает весь размеченный gold-set.
- `MY_CUSTOM_RERANKER_BACKEND` — backend для model id, которого ещё нет в registry. Для обычных Hugging Face cross-encoder моделей оставляйте `CROSS_ENCODER_BACKEND`.

Переменные окружения `DEDUP_RERANKER_BENCHMARK_MODELS`, `DEDUP_RERANKER_BENCHMARK_MAX_PAIRS` и `DEDUP_RERANKER_BENCHMARK_BACKEND` нужны только для запуска из терминала; если они заданы, они переопределяют значения из code-ячейки.

После запуска ячейка показывает таблицу: что вы попросили, какой alias реально резолвится, какой model id будет скачан, какой backend используется и где лежит cache.


In [ ]:
from research.dedup import CrossEncoderMatcher, JinaRerankerMatcher
from research.dedup.matchers.cross_encoder import CrossEncoderConfig
from research.dedup.matchers.jina_reranker import JinaRerankerConfig

# === НАСТРОЙКИ ПОЛЬЗОВАТЕЛЯ ===
# Сюда вписывайте свои reranker-модели для общего benchmark.
# Можно писать короткие aliases: "bge_m3", "qwen3_4b", "qwen3_0_6b", "jina_v3".
# Можно писать полные Hugging Face ids: "BAAI/bge-reranker-v2-m3", "Qwen/Qwen3-Reranker-4B".
MY_RERANKER_MODELS = [
    "bge_m3",
    # "qwen3_4b",  # 4B грузится на CPU, чтобы не падать с MPS out of memory.
    # "qwen3_0_6b",  # более лёгкий Qwen для быстрого smoke-прогона.
    # "jina_v3",
    # "cross-encoder/ms-marco-MiniLM-L6-v2",  # пример своего cross-encoder id
]

# 120 = быстрый пробный срез. Для первого CPU-запуска Qwen-4B можно поставить 12-30. 0 = весь размеченный gold-set.
MY_RERANKER_MAX_PAIRS = 120

# Для своих Hugging Face cross-encoder ids оставляйте CROSS_ENCODER_BACKEND.
# Для Jina-like моделей с методом .rerank используйте TRANSFORMERS_AUTO_MODEL_BACKEND.
# Если хотите запрещать неизвестные ids, поставьте None.
MY_CUSTOM_RERANKER_BACKEND = CROSS_ENCODER_BACKEND
# === КОНЕЦ НАСТРОЕК ПОЛЬЗОВАТЕЛЯ ===


_reranker_models_env = os.environ.get("DEDUP_RERANKER_BENCHMARK_MODELS")
if _reranker_models_env is None:
    RERANKER_BENCHMARK_MODELS = [str(model).strip() for model in MY_RERANKER_MODELS if str(model).strip()]
    RERANKER_BENCHMARK_MODELS_SOURCE = "notebook: MY_RERANKER_MODELS"
else:
    RERANKER_BENCHMARK_MODELS = [model.strip() for model in _reranker_models_env.split(",") if model.strip()]
    RERANKER_BENCHMARK_MODELS_SOURCE = "env: DEDUP_RERANKER_BENCHMARK_MODELS"

_reranker_max_pairs_env = os.environ.get("DEDUP_RERANKER_BENCHMARK_MAX_PAIRS")
if _reranker_max_pairs_env is None:
    RERANKER_BENCHMARK_MAX_PAIRS = int(MY_RERANKER_MAX_PAIRS)
    RERANKER_BENCHMARK_MAX_PAIRS_SOURCE = "notebook: MY_RERANKER_MAX_PAIRS"
else:
    RERANKER_BENCHMARK_MAX_PAIRS = int(_reranker_max_pairs_env)
    RERANKER_BENCHMARK_MAX_PAIRS_SOURCE = "env: DEDUP_RERANKER_BENCHMARK_MAX_PAIRS"

_custom_backend_env = os.environ.get("DEDUP_RERANKER_BENCHMARK_BACKEND")
CUSTOM_RERANKER_BACKEND = _custom_backend_env.strip() if _custom_backend_env is not None else MY_CUSTOM_RERANKER_BACKEND
if isinstance(CUSTOM_RERANKER_BACKEND, str) and CUSTOM_RERANKER_BACKEND.strip().lower() in {"", "none", "null"}:
    CUSTOM_RERANKER_BACKEND = None

RERANKER_BENCHMARK_SUMMARY_PATH = DATA_DIR / "reranker_benchmark_summary_sauces.csv"
RERANKER_BENCHMARK_PREDICTIONS_PATH = DATA_DIR / "reranker_benchmark_predictions_sauces.csv"
ALL_MODEL_BENCHMARK_SUMMARY_PATH = DATA_DIR / "all_model_benchmark_summary_sauces.csv"
ALL_MODEL_BENCHMARK_PREDICTIONS_PATH = DATA_DIR / "all_model_benchmark_predictions_sauces.csv"

print(f"Models source: {RERANKER_BENCHMARK_MODELS_SOURCE}")
print(f"Requested reranker models: {RERANKER_BENCHMARK_MODELS or '[]'}")
print(f"Max pairs source: {RERANKER_BENCHMARK_MAX_PAIRS_SOURCE}; value={RERANKER_BENCHMARK_MAX_PAIRS or 'all'}")
print(f"Custom model backend: {CUSTOM_RERANKER_BACKEND or 'disabled'}")


def _fusion_from_model_spec(spec):
    threshold_high = spec.fusion_threshold_high if spec.fusion_threshold_high is not None else 0.5
    threshold_low = spec.fusion_threshold_low if spec.fusion_threshold_low is not None else 0.2
    return FusionConfig(threshold_high=threshold_high, threshold_low=threshold_low)


def _resolve_reranker_specs(model_inputs, *, custom_backend=None):
    specs = []
    errors = []
    input_by_alias = {}
    allowed_backends = {CROSS_ENCODER_BACKEND, TRANSFORMERS_AUTO_MODEL_BACKEND}
    if custom_backend not in allowed_backends | {None}:
        errors.append({"model_input": "MY_CUSTOM_RERANKER_BACKEND", "error": f"unsupported custom backend: {custom_backend}"})
        custom_backend = None

    for model_input in model_inputs:
        model_input = str(model_input).strip()
        if not model_input:
            continue
        try:
            spec = MODEL_MANAGER.resolve(model_input)
        except Exception as exc:
            if custom_backend is None:
                errors.append({
                    "model_input": model_input,
                    "error": f"{exc}; set MY_CUSTOM_RERANKER_BACKEND for custom model ids",
                })
                continue
            try:
                spec = MODEL_MANAGER.resolve(model_input, backend=custom_backend)
            except Exception as fallback_exc:
                errors.append({
                    "model_input": model_input,
                    "error": f"{exc}; custom backend {custom_backend}: {fallback_exc}",
                })
                continue
        if spec.backend not in allowed_backends:
            errors.append({"model_input": model_input, "error": f"unsupported backend for reranker benchmark: {spec.backend}"})
            continue
        specs.append(spec)
        input_by_alias[spec.alias] = model_input
    return specs, errors, input_by_alias


reranker_model_specs, reranker_model_errors, reranker_model_inputs = _resolve_reranker_specs(
    RERANKER_BENCHMARK_MODELS,
    custom_backend=CUSTOM_RERANKER_BACKEND,
)

benchmark_config_rows = [
    {
        "input": reranker_model_inputs.get(spec.alias, spec.alias),
        "alias": spec.alias,
        "method": spec.method_name or spec.alias,
        "backend": spec.backend,
        "model": spec.model_name,
        "batch_size": spec.batch_size or "",
        "device": spec.device or "auto",
        "documents_per_query": spec.documents_per_query or "",
        "max_pairs": RERANKER_BENCHMARK_MAX_PAIRS or "all",
        "cache_dir": str(MODEL_MANAGER.cache_dir),
        "local_only": MODEL_MANAGER.local_files_only,
    }
    for spec in reranker_model_specs
]
display(pd.DataFrame(benchmark_config_rows))
if reranker_model_errors:
    display(pd.DataFrame(reranker_model_errors))

reranker_benchmark_matchers = []
for spec in reranker_model_specs:
    method_name = spec.method_name or spec.alias
    if spec.backend == CROSS_ENCODER_BACKEND:
        reranker_benchmark_matchers.append(
            CrossEncoderMatcher(
                CrossEncoderConfig(
                    model_name=spec.alias,
                    method_name=method_name,
                    batch_size=spec.batch_size or 1,
                    device=spec.device,
                    trust_remote_code=spec.trust_remote_code,
                    prompts=spec.prompts,
                    default_prompt_name=spec.default_prompt_name,
                    fusion=_fusion_from_model_spec(spec),
                )
            )
        )
    elif spec.backend == TRANSFORMERS_AUTO_MODEL_BACKEND:
        reranker_benchmark_matchers.append(
            JinaRerankerMatcher(
                JinaRerankerConfig(
                    model_name=spec.alias,
                    method_name=method_name,
                    documents_per_query=spec.documents_per_query or 8,
                    trust_remote_code=spec.trust_remote_code,
                    fusion=_fusion_from_model_spec(spec),
                )
            )
        )

if not reranker_benchmark_matchers:
    display(pd.DataFrame([{
        "status": "no_models_selected",
        "hint": "add models to MY_RERANKER_MODELS, for example bge_m3, qwen3_4b or jina_v3",
    }]))
else:
    display(pd.DataFrame([
        {"method": matcher.name, "status": matcher.status().message}
        for matcher in reranker_benchmark_matchers
    ]))


## Блок кода 17. Запуск новых reranker-моделей

Эта ячейка считает score только для новых тяжёлых моделей из `MY_RERANKER_MODELS` после возможного env override. Старые методы выше уже посчитаны, поэтому здесь они не запускаются повторно.

Если `MY_RERANKER_MAX_PAIRS > 0`, берётся небольшой сбалансированный срез по `dev/test` и классам. Такой срез годится для проверки, что модель запускается. Для финального выбора лучшего решения поставьте `MY_RERANKER_MAX_PAIRS = 0` в предыдущей ячейке.


In [ ]:
def _benchmark_frame(frame: pd.DataFrame, max_pairs: int) -> pd.DataFrame:
    if frame.empty or max_pairs <= 0 or len(frame) <= max_pairs:
        return _with_benchmark_pair_key(frame)
    ordered = frame.copy()
    ordered["_round_robin_order"] = ordered.groupby(["eval_split", "same_base_product"]).cumcount()
    sampled = (
        ordered.sort_values(["_round_robin_order", "eval_split", "label"])
        .head(max_pairs)
        .drop(columns=["_round_robin_order"])
        .sort_index()
    )
    return _with_benchmark_pair_key(sampled.reset_index(drop=True))


reranker_benchmark_pairs = _benchmark_frame(labeled_pairs, RERANKER_BENCHMARK_MAX_PAIRS)
reranker_benchmark_payloads: dict[str, dict[str, object]] = {}
reranker_benchmark_status_rows: list[dict[str, object]] = []

if reranker_benchmark_pairs.empty:
    print("Benchmark skipped: gold-set is empty.")
else:
    print(f"Benchmark pairs: {len(reranker_benchmark_pairs)} / {len(labeled_pairs)}")
    for matcher in reranker_benchmark_matchers:
        started = time.perf_counter()
        scores, status = _score_matcher(matcher, reranker_benchmark_pairs)
        elapsed = time.perf_counter() - started
        reranker_benchmark_status_rows.append(
            {
                "method": matcher.name,
                "model": getattr(matcher.config, "model_name", ""),
                "status": status,
                "seconds": round(elapsed, 3),
                "pairs": len(reranker_benchmark_pairs),
                "seconds_per_pair": round(elapsed / len(reranker_benchmark_pairs), 4) if len(reranker_benchmark_pairs) else 0.0,
            }
        )
        if status == "ready":
            reranker_benchmark_payloads[matcher.name] = {
                "matcher": matcher,
                "scores": scores,
                "seconds": elapsed,
                "pairs": reranker_benchmark_pairs,
            }

if reranker_benchmark_status_rows:
    display(pd.DataFrame(reranker_benchmark_status_rows))

## Блок кода 18. Cost-sensitive threshold calibration всех методов

Здесь собирается единая score-таблица для всех доступных methods и калибруются два порога на `dev`:

- `threshold_auto_same`: максимально recall-овый auto-merge порог при `auto_same_precision >= TARGET_AUTO_SAME_PRECISION` и `false_merge_count <= MAX_FALSE_MERGES_ON_DEV`.
- `threshold_auto_diff`: максимально покрывающий auto-reject порог при `auto_diff_precision >= TARGET_AUTO_DIFF_PRECISION`.

`test` используется только для финальной проверки выбранных на dev порогов.


In [ ]:
def _collect_all_method_scores() -> pd.DataFrame:
    frames: list[pd.DataFrame] = []
    benchmark_keys = set(reranker_benchmark_pairs["benchmark_pair_key"]) if not reranker_benchmark_pairs.empty else set()

    for method, payload in scored_methods.items():
        frame = _score_frame(method, labeled_pairs, payload["scores"], benchmark_source="baseline_or_embedding")
        frames.append(frame)

    if cross_encoder_payload is not None:
        matcher = cross_encoder_payload["matcher"]
        frames.append(
            _score_frame(matcher.name, labeled_pairs, cross_encoder_payload["scores"], benchmark_source="cross_encoder")
        )

    for method, payload in reranker_benchmark_payloads.items():
        frames.append(
            _score_frame(method, payload["pairs"], payload["scores"], benchmark_source="reranker")
        )

    if not frames:
        return pd.DataFrame()

    combined = pd.concat(frames, ignore_index=True)
    if benchmark_keys:
        combined = combined[combined["benchmark_pair_key"].isin(benchmark_keys)].copy()
    return combined.reset_index(drop=True)


def _method_slug(method: str) -> str:
    return "".join(char if char.isalnum() or char in {"_", "-"} else "_" for char in method).strip("_") or "method"


def _plot_threshold_diagnostics(results: dict[str, pd.DataFrame], predictions: pd.DataFrame, reports_dir: Path) -> None:
    if plt is None or predictions.empty:
        print("Plot export skipped: matplotlib is not installed or predictions are empty.")
        return

    reports_dir.mkdir(parents=True, exist_ok=True)
    same_grid = results.get("auto_same_threshold_grid", pd.DataFrame())
    calibration = results.get("calibration_on_dev", pd.DataFrame())

    for method, method_frame in predictions.groupby("method", sort=False):
        slug = _method_slug(str(method))
        method_calibration = calibration[calibration["method"].eq(method)]
        selected_diff = None
        if not method_calibration.empty:
            selected_diff = float(method_calibration.iloc[0]["threshold_auto_diff"])

        fig, ax = plt.subplots(figsize=(8, 4))
        for target_value, label in [(0, "different_product"), (1, "same_base_product")]:
            scores = method_frame[method_frame["same_base_product"].eq(target_value)]["score"].dropna()
            if not scores.empty:
                ax.hist(scores, bins=20, alpha=0.55, label=label)
        ax.set_title(f"Score distribution: {method}")
        ax.set_xlabel("score")
        ax.set_ylabel("pairs")
        ax.legend()
        fig.tight_layout()
        fig.savefig(reports_dir / f"score_distribution_{slug}.png", dpi=150)
        plt.close(fig)

        method_grid = same_grid[same_grid["method"].eq(method)].copy()
        if method_grid.empty:
            continue

        fig, ax = plt.subplots(figsize=(6, 5))
        ax.plot(method_grid["auto_same_recall"], method_grid["auto_same_precision"], marker="o", linewidth=1)
        ax.set_title(f"Precision-recall: {method}")
        ax.set_xlabel("auto_same_recall")
        ax.set_ylabel("auto_same_precision")
        ax.set_ylim(0, 1.05)
        fig.tight_layout()
        fig.savefig(reports_dir / f"precision_recall_{slug}.png", dpi=150)
        plt.close(fig)

        fig, ax = plt.subplots(figsize=(8, 4))
        ax.plot(method_grid["threshold"], method_grid["false_merge_count"], marker="o", linewidth=1)
        ax.set_title(f"Threshold vs false merges: {method}")
        ax.set_xlabel("threshold_auto_same")
        ax.set_ylabel("false_merge_count on dev")
        fig.tight_layout()
        fig.savefig(reports_dir / f"threshold_false_merges_{slug}.png", dpi=150)
        plt.close(fig)

        if selected_diff is not None:
            manual_rows = []
            dev_frame = method_frame[method_frame["eval_split"].eq("dev")].copy()
            for threshold in method_grid["threshold"]:
                auto_same = dev_frame["score"].ge(threshold)
                auto_diff = dev_frame["score"].le(selected_diff) & ~auto_same
                manual_rate = 1.0 - ((auto_same.sum() + auto_diff.sum()) / len(dev_frame) if len(dev_frame) else 0.0)
                manual_rows.append({"threshold": threshold, "manual_review_rate": manual_rate})
            manual_grid = pd.DataFrame(manual_rows)
            fig, ax = plt.subplots(figsize=(8, 4))
            ax.plot(manual_grid["threshold"], manual_grid["manual_review_rate"], marker="o", linewidth=1)
            ax.set_title(f"Threshold vs manual review: {method}")
            ax.set_xlabel("threshold_auto_same")
            ax.set_ylabel("manual_review_rate on dev")
            ax.set_ylim(0, 1.05)
            fig.tight_layout()
            fig.savefig(reports_dir / f"threshold_manual_review_{slug}.png", dpi=150)
            plt.close(fig)


all_method_scores = _collect_all_method_scores()
if all_method_scores.empty:
    threshold_results = {
        "calibration_on_dev": pd.DataFrame(),
        "evaluation_on_test": pd.DataFrame(),
        "predictions": pd.DataFrame(),
        "auto_same_threshold_grid": pd.DataFrame(),
        "auto_diff_threshold_grid": pd.DataFrame(),
    }
    display(pd.DataFrame([{"status": "no_ready_method_scores"}]))
else:
    threshold_results = calibrate_and_evaluate_methods(all_method_scores, config=THRESHOLD_CONFIG)
    calibration_on_dev = threshold_results["calibration_on_dev"].sort_values(
        ["passed_auto_same_constraints", "auto_coverage", "auto_same_recall"],
        ascending=[False, False, False],
    ).reset_index(drop=True)
    evaluation_on_test = threshold_results["evaluation_on_test"].sort_values(
        ["passed_auto_same_constraints", "auto_coverage", "auto_same_precision"],
        ascending=[False, False, False],
    ).reset_index(drop=True)
    cost_sensitive_predictions = threshold_results["predictions"].copy()
    cost_sensitive_predictions["mode"] = "cost_sensitive_calibrated"

    display(calibration_on_dev)
    display(evaluation_on_test)

    confusion_rows = []
    for method in cost_sensitive_predictions["method"].drop_duplicates():
        for split_name in ["dev", "test"]:
            part = cost_sensitive_predictions[
                cost_sensitive_predictions["method"].eq(method)
                & cost_sensitive_predictions["eval_split"].eq(split_name)
            ].copy()
            if part.empty:
                continue

            forced_cm = forced_confusion_matrix(part)
            triage_cm = triage_confusion_matrix(part)
            print(f"Forced confusion matrix: {method} / {split_name}")
            display(forced_cm)
            print(f"Triage confusion matrix: {method} / {split_name}")
            display(triage_cm)

            for matrix_name, matrix in [("forced", forced_cm), ("triage", triage_cm)]:
                long_matrix = matrix.rename_axis("true_label").reset_index().melt(
                    id_vars="true_label",
                    var_name="predicted_label",
                    value_name="count",
                )
                long_matrix["method"] = method
                long_matrix["eval_split"] = split_name
                long_matrix["matrix_type"] = matrix_name
                confusion_rows.append(long_matrix)

    if confusion_rows:
        pd.concat(confusion_rows, ignore_index=True).to_csv(
            REPORTS_DIR / "threshold_confusion_matrices.csv",
            index=False,
        )

    write_threshold_reports(threshold_results, REPORTS_DIR)
    _plot_threshold_diagnostics(threshold_results, cost_sensitive_predictions, REPORTS_DIR)

    calibration_on_dev.to_csv(REPORTS_DIR / "threshold_calibration_dev.csv", index=False)
    evaluation_on_test.to_csv(REPORTS_DIR / "threshold_evaluation_test.csv", index=False)
    threshold_results["auto_same_threshold_grid"].to_csv(REPORTS_DIR / "auto_same_threshold_grid_dev.csv", index=False)
    threshold_results["auto_diff_threshold_grid"].to_csv(REPORTS_DIR / "auto_diff_threshold_grid_dev.csv", index=False)

    summary_export = pd.concat(
        [
            calibration_on_dev.assign(mode="calibration_on_dev"),
            evaluation_on_test.assign(mode="evaluation_on_test"),
        ],
        ignore_index=True,
    )
    summary_export.to_csv(SUMMARY_PATH, index=False)
    cost_sensitive_predictions.to_csv(PREDICTIONS_PATH, index=False)
    cost_sensitive_predictions[cost_sensitive_predictions["false_merge"]].to_csv(FALSE_MERGES_PATH, index=False)

    ALL_MODEL_BENCHMARK_SUMMARY_PATH = DATA_DIR / "all_model_benchmark_summary_sauces.csv"
    ALL_MODEL_BENCHMARK_PREDICTIONS_PATH = DATA_DIR / "all_model_benchmark_predictions_sauces.csv"
    summary_export.to_csv(ALL_MODEL_BENCHMARK_SUMMARY_PATH, index=False)
    cost_sensitive_predictions.to_csv(ALL_MODEL_BENCHMARK_PREDICTIONS_PATH, index=False)

    print(f"Saved calibration: {REPORTS_DIR / 'threshold_calibration_dev.csv'}")
    print(f"Saved evaluation: {REPORTS_DIR / 'threshold_evaluation_test.csv'}")
    print(f"Saved predictions: {PREDICTIONS_PATH} ({len(cost_sensitive_predictions)} rows)")
    print(f"Saved summary: {SUMMARY_PATH} ({len(summary_export)} rows)")


## Как читать общий бенчмарк

Для production auto-merge смотри сначала `artifacts/reports/threshold_calibration_dev.csv`, а не строки с `macro_f1`.

Главный критерий auto-merge:

- `auto_same_precision >= TARGET_AUTO_SAME_PRECISION`;
- `false_merge_count <= MAX_FALSE_MERGES_ON_DEV` на dev;
- среди прошедших threshold-ов выбирается максимальный `auto_same_recall`.

`macro-F1` не является главным production-критерием, потому что он симметрично награждает баланс классов, а в SKU dedup false merge разных товаров стоит намного дороже, чем пропуск дубля.

Для каждой модели смотри:

- `passed_auto_same_constraints` — можно ли вообще доверять auto-merge этой модели при текущих ограничениях;
- `manual_review_rate` — какая доля пар ушла в ручную проверку;
- `auto_coverage` — какая доля пар получила автоматическое решение;
- `false_merges_on_test.csv` и `manual_review_pairs_test.csv` — конкретные пары для чтения глазами.

Test split не используется для выбора threshold или модели. Он нужен только для финальной проверки выбранных на dev порогов.
